# Chimeric RNA-Seq Negative Data Pipeline (Enlarged)

This notebook implements a full pipeline to produce labeled **False Negative** and **False Positive** training data for a chimera detection model.

## 1. Terminology & Strategy

Both datasets generated here will be labeled as **False (0)**, but they serve different purposes:

* **False Negative Candidates (Baseline / Canonical):**
    * **Content:** Real, high-quality human transcripts (from UniProt Swiss-Prot).
    * **Goal:** Teach the model what "normal" RNA looks like so it doesn't flag healthy tissue.
    * **Label:** 0

* **False Positive Candidates (Hard Negatives / Synthetic):**
    * **Content:** Synthetic sequences designed to trick the model (Reversed, Shuffled, and Randomly Paired).
    * **Goal:** Teach the model to distinguish specific fusion breakpoints from random noise or artifacts.
    * **Label:** 0

## 2. Setup & Configuration

We will use **Biopython** to handle sequence processing. Ensure it is installed (`pip install biopython`).

In [ ]:
import os
import gzip
import urllib.request
import random
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

# ============ CONFIGURATION ============

# Source: UniProt Swiss-Prot (Reviewed Canonical Sequences)
UNIPROT_URL = "https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/uniprot_sprot.fasta.gz"

# Output Filenames
FP_OUTPUT_FILE = "false_positive_candidates.fasta"  # 'Hard' negatives (Artifacts/Synthetic)
FN_OUTPUT_FILE = "false_negative_candidates.fasta"  # 'Easy' negatives (Canonical Real RNA)

# Parameters
SAMPLE_SIZE = 5000  # INCREASED to 5000 samples
SEED = 42

print("✅ Pipeline configured.")

## 3. Data Downloader

This step downloads the official UniProt Swiss-Prot database (gzipped) if it doesn't already exist locally.

In [ ]:
def download_data(url, local_filename):
    """Downloads a file from a URL if it doesn't already exist."""
    if not os.path.exists(local_filename):
        print(f"Downloading {local_filename}...")
        try:
            urllib.request.urlretrieve(url, local_filename)
            print("Download complete.")
        except Exception as e:
            print(f"Error downloading: {e}")
            return False
    else:
        print(f"File {local_filename} already exists. Skipping download.")
    return True

# Download the UniProt fasta (gzipped)
local_gz_file = "uniprot_sprot.fasta.gz"
success = download_data(UNIPROT_URL, local_gz_file)

if not success:
    raise Exception("Failed to download necessary data.")

## 4. Load Canonical Sequences

We parse the downloaded file, filtering specifically for human sequences (*Homo sapiens*) to create our "Real" baseline.

In [ ]:
def load_canonical_sequences(gz_file, limit=None):
    """Parses the Gzipped FASTA file and returns a list of SeqRecords."""
    seqs = []
    print("Parsing sequences...")
    # Open the gzipped file in text mode
    with gzip.open(gz_file, "rt") as handle:
        for i, record in enumerate(SeqIO.parse(handle, "fasta")):
            # Filter for Human sequences
            if "Homo sapiens" in record.description: 
                seqs.append(record)
            
            # Stop if we reach the sample limit
            if limit and len(seqs) >= limit:
                break
    print(f"✅ Loaded {len(seqs)} canonical human sequences.")
    return seqs

# Load real biological sequences
real_transcripts = load_canonical_sequences(local_gz_file, limit=SAMPLE_SIZE)

## 5. Generate Synthetic Datasets (Optimized Size)

Here we create the two distinct datasets:
1. **False Negative Set:** Direct copies of the real transcripts (5000 samples).
2. **False Positive Set:** Artificially manipulated sequences.
   * **Logic:** To keep the dataset size between 5000-7500, we will randomly select **1 or 2 synthetic strategies** (Reverse, Shuffle, or Random Pair) for each real transcript, rather than generating all 3 for everyone.

In [ ]:
random.seed(SEED)

# --- A. "False Negative" Dataset (Baseline / Canonical) ---
# These are just the real, normal transcripts.
fn_records = []
for record in real_transcripts:
    new_id = f"NEG_CANONICAL_{record.id}"
    new_rec = SeqRecord(
        record.seq,
        id=new_id,
        description="Label:0 source:UniProt_Canonical"
    )
    fn_records.append(new_rec)

# --- B. "False Positive" Dataset (Synthetic / Hard Negatives) ---
fp_records = []

# Synthetic strategies available
strategies = ['reverse', 'shuffle', 'random_pair']

for record in real_transcripts:
    seq_str = str(record.seq)
    
    # Randomly pick how many variants to make for this sequence (1 or 2)
    # This ensures total size is ~1.5x the original (approx 7500)
    num_variants = random.choice([1, 2])
    
    # Randomly pick which strategies to use
    chosen_strategies = random.sample(strategies, num_variants)
    
    for strat in chosen_strategies:
        if strat == 'reverse':
            # 1. Reverse Sequence (Tests directionality)
            rev_seq = seq_str[::-1]
            fp_records.append(
                SeqRecord(
                    Seq(rev_seq),
                    id=f"NEG_SYNTH_REVERSE_{record.id}",
                    description="Label:0 type:reversed"
                )
            )
            
        elif strat == 'shuffle':
            # 2. Shuffled Sequence (Tests motif dependency vs composition)
            seq_list = list(seq_str)
            random.shuffle(seq_list)
            shuffled_seq = "".join(seq_list)
            fp_records.append(
                SeqRecord(
                    Seq(shuffled_seq),
                    id=f"NEG_SYNTH_SHUFFLED_{record.id}",
                    description="Label:0 type:shuffled"
                )
            )
            
        elif strat == 'random_pair':
            # 3. Random Pair "Fusion" (Tests structural joins)
            other_record = random.choice(real_transcripts)
            half_len_1 = len(seq_str) // 2
            half_len_2 = len(other_record.seq) // 2
            
            hybrid_seq = seq_str[:half_len_1] + str(other_record.seq)[half_len_2:]
            fp_records.append(
                SeqRecord(
                    Seq(hybrid_seq),
                    id=f"NEG_SYNTH_RANDOMPAIR_{record.id}_vs_{other_record.id}",
                    description="Label:0 type:random_fusion_artifact"
                )
            )

print(f"✅ Generated {len(fn_records)} Canonical Negative samples.")
print(f"✅ Generated {len(fp_records)} Synthetic False Positive samples.")

## 6. Save to FASTA

Write the records to disk.

In [ ]:
print(f"Writing to {FN_OUTPUT_FILE}...")
SeqIO.write(fn_records, FN_OUTPUT_FILE, "fasta")

print(f"Writing to {FP_OUTPUT_FILE}...")
SeqIO.write(fp_records, FP_OUTPUT_FILE, "fasta")

print("\n🎉 Processing Complete!")
print(f"1. [False Negative Candidates] -> {os.path.abspath(FN_OUTPUT_FILE)}")
print(f"2. [False Positive Candidates] -> {os.path.abspath(FP_OUTPUT_FILE)}")